# Artists data analysis

Let's assert tracks data analysis by taking a first inspection of the artists dataset

In [1]:
import pandas as pd
import altair as alt
from os import path
import warnings
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from IPython.display import display, Markdown
from tqdm import tqdm
import time
import requests

#unique user_agent to identify your project to OSM
osm_locator = Nominatim(user_agent="artist_data_analysis_v1", timeout=10) #Nominatim = search engine for OpenStreetMap

#rate-limited to respect OSM's free usage policy
geocode_osm = RateLimiter(osm_locator.geocode, min_delay_seconds=1.0)
reverse_osm = RateLimiter(osm_locator.reverse, min_delay_seconds=1.0)

tqdm.pandas()

dataset_path = path.join('..', 'dataset', 'artists.csv')
df = pd.read_csv(dataset_path, sep=';')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_author     104 non-null    object 
 1   name          104 non-null    object 
 2   gender        104 non-null    object 
 3   birth_date    73 non-null     object 
 4   birth_place   72 non-null     object 
 5   nationality   71 non-null     object 
 6   description   86 non-null     object 
 7   active_start  50 non-null     object 
 8   active_end    0 non-null      float64
 9   province      70 non-null     object 
 10  region        68 non-null     object 
 11  country       70 non-null     object 
 12  latitude      72 non-null     float64
 13  longitude     72 non-null     float64
dtypes: float64(3), object(11)
memory usage: 11.5+ KB


validation helper function checking expected types validity

In [3]:
def check_type_validity(value, expected_type):
    return not isinstance(value, expected_type)

before the analysis, let's strip the string columns from any invisible unicode characters:

In [4]:
cols_to_strip = [
    "id_author", "name", "gender", "birth_date",
    "birth_place", "nationality", "description",
    "active_start", "province", "region", "country"
]

zero_width_pattern = r"[\u200b\u200c\u200d\uFEFF]"

nbsp_pattern = r"[\xa0]"

for col in cols_to_strip:
    if col in df.columns:
        #strip only non empty rows
        non_null_mask = df[col].notna()
        
        df.loc[non_null_mask, col] = (
            df.loc[non_null_mask, col]
            .astype(str)
            .str.replace(zero_width_pattern, "", regex=True)
            .str.replace(nbsp_pattern, " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

print("string columns have been stripped and cleaned")

string columns have been stripped and cleaned


### id_author

In [5]:
invalid_elems = df[df['id_author'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['id_author'])

nan_indexes = df.index[df['id_author'].isna()].tolist()
print(f"number of missing values in id_author: {len(nan_indexes)}")

unique_ids = df['id_author'].nunique()
print(f"number of unique id_authors: {unique_ids}")

Series([], Name: id_author, dtype: object)
number of missing values in id_author: 0
number of unique id_authors: 104


as there seems to be no issues related to the id uniqueness of the form "ART{8-digit-number}", a merge between this and the tracks dataset will be performed

In [6]:
tracks_path = path.join('..', 'dataset', 'cleaned_tracks.csv')
df_tracks = pd.read_csv(tracks_path, sep=',')

merged_df = pd.merge(
    df, 
    df_tracks, 
    left_on='id_author', 
    right_on='id_artist', 
    how='inner'
)

### name

In [7]:
invalid_elems = df[df['name'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['name'])

nan_indexes = df.index[df['name'].isna()].tolist()
print(f"number of missing values in name: {len(nan_indexes)}")

Series([], Name: name, dtype: object)
number of missing values in name: 0


In [8]:
artist_names = merged_df['name'].astype(str).str.lower()
track_artist_names = merged_df['name_artist'].astype(str).str.lower()

mismatches = merged_df[artist_names != track_artist_names]

if mismatches.empty:
    print("All artist names match between the artist and tracks datasets")
else:
    unique_mismatch_examples = mismatches.drop_duplicates(subset=['id_author'])
    
    print(f"Found {len(unique_mismatch_examples)} artists with mismatched names, affecting {len(mismatches)} tracks")
    
    cols_to_show = ['id_author', 'name', 'name_artist']
    print(unique_mismatch_examples[cols_to_show])

Found 10 artists with mismatched names, affecting 872 tracks
        id_author              name     name_artist
344   ART64265460         anna pepe            ANNA
1598  ART67409252  chadia rodriguez          Chadia
2396  ART63985757    dargen d_amico  Dargen D’Amico
4986  ART04141409       guè pequeno             Guè
5957  ART88199433       joey funboy      Joey (ITA)
6988  ART37807199            mike24        Highsnob
7064  ART43601431         miss keta       M¥SS KETA
7624  ART71846481          mr. rain         Mr.Rain
8520  ART42220690            o zulù         ’O Zulù
9502  ART56967402      samuel heron    Samuel Costa


the mismatches are related to the name form, there are no errors. The column 'name' seems to be more normalized and complete compared to the column 'name_artist'. Rhe only discrepancy is with ART37807199, where the name_artist value (Highsnob) is more 'accurate', as mike24 is a pseudonym of the same person.

### gender

In [9]:
invalid_elems = df[df['gender'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['gender'])

nan_indexes = df.index[df['gender'].isna()].tolist()
print(f"number of missing values in gender: {len(nan_indexes)}")

Series([], Name: gender, dtype: object)
number of missing values in gender: 0


In [10]:
print(df['gender'].value_counts(dropna=False))

gender
M    87
F    17
Name: count, dtype: int64


### birth_date

In [11]:
invalid_elems = df[df['birth_date'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['birth_date'].head(10))

nan_indexes = df.index[df['birth_date'].isna()].tolist()
print(f"number of missing values in birth date: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: birth_date, dtype: object
number of missing values in birth date: 31


In [12]:
dates_converted = pd.to_datetime(df['birth_date'], errors='coerce')

malformed_mask = df['birth_date'].notna() & dates_converted.isna()
malformed_rows = df[malformed_mask]
print(f"number of malformed date strings (cannot be parsed, will be set to NaT): {len(malformed_rows)}")

if not malformed_rows.empty:
    print("malformed dates:")
    print(malformed_rows['birth_date'].unique()[:10])

df['birth_date'] = dates_converted

#checking for syntactically wrong values
now = pd.Timestamp.now()
future_dates = df[df['birth_date'] > now]

if not future_dates.empty:
    print(future_dates[['id_author', 'name', 'birth_date']])

old_date_threshold = pd.Timestamp("1950-01-01")
ancient_dates = df[df['birth_date'] < old_date_threshold]

if not ancient_dates.empty:
    print(ancient_dates[['id_author', 'name', 'birth_date']].head())

number of malformed date strings (cannot be parsed, will be set to NaT): 1
malformed dates:
['http://www.wikidata.org/.well-known/genid/4111f32c49a23235b2e902dc8621d27c']


### birth_place

In [13]:
invalid_elems = df[df['birth_place'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['birth_place'].head(10))

nan_indexes = df.index[df['birth_place'].isna()].tolist()
print(f"number of missing values in birth place: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: birth_place, dtype: object
number of missing values in birth place: 32


In [14]:
unique_places = sorted(df['birth_place'].dropna().unique().tolist())

print(f"number of unique birth places: {len(unique_places)}")
print(unique_places)

number of unique birth places: 40
['Almería', 'Alpignano', 'Avellino', 'Bologna', 'Brescia', 'Buenos Aires', 'Cagliari', 'Desenzano del Garda', 'Firenze', 'Fiumicino', 'Gallarate', 'Genova', 'Grottaglie', 'Grugliasco', 'La Spezia', 'Lodi', 'Milano', 'Napoli', 'Nicosia', 'Nocera Inferiore', 'Olbia', 'Padova', 'Pieve Emanuele', 'Reggio Calabria', 'Rho', 'Roma', 'Salerno', 'San Benedetto del Tronto', 'San Siro', 'Scafati', 'Scampia', 'Senigallia', 'Sesto San Giovanni', 'Singapore', 'Sternatia', 'Torino', 'Treviso', 'Verona', 'Vicenza', 'Vimercate']


### nationality

In [15]:
invalid_elems = df[df['nationality'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['nationality'].head(10))

nan_indexes = df.index[df['nationality'].isna()].tolist()
print(f"number of missing values in nationality: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: nationality, dtype: object
number of missing values in nationality: 33


In [16]:
unique_nationalities = sorted(df['nationality'].dropna().unique().tolist())

print(f"number of unique nationalities: {len(unique_nationalities)}")
print(unique_nationalities)

number of unique nationalities: 2
['Argentina', 'Italia']


### description

In [17]:
invalid_elems = df[df['description'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['description'].head(10))

nan_indexes = df.index[df['description'].isna()].tolist()
print(f"number of missing values in description: {len(nan_indexes)}")

2     NaN
3     NaN
9     NaN
11    NaN
22    NaN
24    NaN
31    NaN
42    NaN
44    NaN
50    NaN
Name: description, dtype: object
number of missing values in description: 18


In [18]:
print(df['description'].head(20))

0                              gruppo musicale italiano
1                  cantautore e rapper italiano (1990-)
2                                                   NaN
3                                                   NaN
4                      gruppo musicale hip hop italiano
5                                     cantante italiano
6                 cantautrice e rapper italiana (1983-)
7     rapper, disc jockey, beatmaker e produttore di...
8                                               cognome
9                                                   NaN
10                                              cognome
11                                                  NaN
12                                              cognome
13    rapper, cantautore e produttore discografico i...
14                              rapper italiano (1998-)
15                              rapper italiana (1998-)
16                     rapper e attore italiano (1982-)
17                             gruppo musicale i

the only relevant information in this column could be the what it looks to be the artist birth year, so it could be use to fill eventually some missing birth_date values.

In [19]:
#looks for '(', followed by exactly 4 digits
regex_pattern = r'\((\d{4})'
extracted_years = df['description'].astype(str).str.extract(regex_pattern)[0]

df['desc_extracted_year'] = pd.to_numeric(extracted_years, errors='coerce')
df['existing_birth_year'] = df['birth_date'].dt.year

check_mask = df['desc_extracted_year'].notna() & df['existing_birth_year'].notna()
matches = df[check_mask & (df['desc_extracted_year'] == df['existing_birth_year'])]
mismatches = df[check_mask & (df['desc_extracted_year'] != df['existing_birth_year'])]

print(f"validation check on existing birth_date:")
print(f"- matches: {len(matches)}")
print(f"- mismatches: {len(mismatches)}")

fill_mask = df['birth_date'].isna() & df['desc_extracted_year'].notna()

num_to_fill = fill_mask.sum()
print(f"\nfound {num_to_fill} artists with missing birth_date that can be filled from description!")

validation check on existing birth_date:
- matches: 59
- mismatches: 0

found 0 artists with missing birth_date that can be filled from description!


as there are no useful information in the description field, it's possible to delete the column.

### active_start

In [20]:
invalid_elems = df[df['active_start'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['active_start'].head(10))

nan_indexes = df.index[df['active_start'].isna()].tolist()
print(f"number of missing values in active start: {len(nan_indexes)}")

2     NaN
3     NaN
5     NaN
8     NaN
10    NaN
11    NaN
12    NaN
14    NaN
15    NaN
16    NaN
Name: active_start, dtype: object
number of missing values in active start: 54


In [21]:
dates_converted = pd.to_datetime(df['active_start'], errors='coerce')

malformed_mask = df['active_start'].notna() & dates_converted.isna()
malformed_rows = df[malformed_mask]
print(f"number of malformed date strings (cannot be parsed, will be set to NaT): {len(malformed_rows)}")

if not malformed_rows.empty:
    print("malformed dates:")
    print(malformed_rows['active_start'].unique()[:10])

df['active_start'] = dates_converted

#checking for syntactically wrong values
now = pd.Timestamp.now()
future_dates = df[df['active_start'] > now]

if not future_dates.empty:
    print(future_dates[['id_author', 'name', 'active_start']])

number of malformed date strings (cannot be parsed, will be set to NaT): 0


as many values are missing, it's possible to obtain an indicative year of start rap activity of the artists from the firrst pulished song in the tracks dataset for that artist.

In [22]:
valid_tracks_mask = df_tracks['year'].notna()
df_tracks_clean = df_tracks.loc[valid_tracks_mask].copy() #only tracks with year

df_tracks_clean['month'] = df_tracks_clean['month'].fillna(1)
df_tracks_clean['day'] = df_tracks_clean['day'].fillna(1) #filling day and month to 1 in case they're missing

df_tracks_clean['year'] = df_tracks_clean['year'].astype(int)
df_tracks_clean['month'] = df_tracks_clean['month'].astype(int)
df_tracks_clean['day'] = df_tracks_clean['day'].astype(int)

df_tracks_clean['release_date_constructed'] = pd.to_datetime(
    df_tracks_clean[['year', 'month', 'day']], 
    errors='coerce'
)

earliest_dates = (
    df_tracks_clean.groupby('id_artist')['release_date_constructed']
    .min()
    .reset_index()
    .rename(columns={'release_date_constructed': 'first_track_release'})
)

merged_df = pd.merge(
    df, 
    earliest_dates, 
    left_on='id_author', 
    right_on='id_artist', 
    how='left'
)

missing_active_mask = merged_df['active_start'].isna()
recoverable_mask = missing_active_mask & merged_df['first_track_release'].notna()

num_recoverable = recoverable_mask.sum()
print(f"number of artists with missing 'active_start' that can be filled from tracks: {num_recoverable}")

if num_recoverable > 0:
    df.loc[recoverable_mask, 'active_start'] = merged_df.loc[recoverable_mask, 'first_track_release']
    print(f"recovered {num_recoverable} 'active_start', filling with 1 the missing month and day values")

number of artists with missing 'active_start' that can be filled from tracks: 54
recovered 54 'active_start', filling with 1 the missing month and day values


as the active start year of the column 'description' doesn't match any of the active_start values, then we can say that 'description' contains no useful information, so it can be deleted.

### active_end

In [23]:
invalid_elems = df[df['active_end'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['active_end'].head(10))

nan_indexes = df.index[df['active_end'].isna()].tolist()
print(f"number of missing values in active end: {len(nan_indexes)}")

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: active_end, dtype: float64
number of missing values in active end: 104


as no one of those artists did stop his/her career, we can safely delete this column as well.

### province

In [24]:
invalid_elems = df[df['province'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['province'].head(10))

nan_indexes = df.index[df['province'].isna()].tolist()
print(f"number of missing values in province: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: province, dtype: object
number of missing values in province: 34


In [25]:
unique_provinces = sorted(df['province'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_provinces)}")
print(unique_provinces)

number of unique provinces: 26
['Ancona', 'Ascoli Piceno', 'Avellino', 'Bologna', 'Brescia', 'Cagliari', 'Enna', 'Firenze', 'Gallura', 'Genova', 'La Spezia', 'Lecce', 'Lodi', 'Milano', 'Monza e della Brianza', 'Napoli', 'Padova', 'Reggio Calabria', 'Roma', 'Salerno', 'Taranto', 'Torino', 'Treviso', 'Varese', 'Verona', 'Vicenza']


### region

In [26]:
invalid_elems = df[df['region'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['region'].head(10))

nan_indexes = df.index[df['region'].isna()].tolist()
print(f"number of missing values in region: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: region, dtype: object
number of missing values in region: 36


In [27]:
unique_regions = sorted(df['region'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_regions)}")
print(unique_regions)

number of unique provinces: 13
['Calabria', 'Campania', 'Emilia-Romagna', 'Lazio', 'Liguria', 'Lombardia', 'Marche', 'Piemonte', 'Puglia', 'Sardegna', 'Sicilia', 'Toscana', 'Veneto']


it's possible to fill the region based on the province, so those rows missing the region but not the province will be filled.

In [28]:
mask_missing_region = df['region'].isna() & df['province'].notna()
print(f"rows missing Region but having province: {mask_missing_region.sum()}")

#unique list of provinces that need a region lookup, for a more general applicable solution
provinces_to_lookup = df.loc[mask_missing_region, 'province'].unique()

def find_region_for_province(province_name):
    try:
        query = f"{province_name}, Italy"
        
        location = geocode_osm(query, addressdetails=True, language='it')
        
        if location and 'address' in location.raw:
            addr = location.raw['address']
            
            # --- DEBUG PRINT: What keys did OSM actually return? ---
            # This is crucial. Maybe it calls it 'region' instead of 'state'?
            # We print this for the first few to avoid spamming, or if state is missing.
            found_state = addr.get('state')
            
            if found_state is None:
                print(f"   [DEBUG] Found '{province_name}' but 'state' key is missing.")
                print(f"           Available keys: {list(addr.keys())}")
            
            return found_state
        else:
            print(f"   [DEBUG] OSM returned NO location for query: '{query}'")
            return None
    except Exception as e:
        print(f"   [ERROR] Exception for '{province_name}': {e}")
        return None

province_region_map = {}

if len(provinces_to_lookup) > 0:
    for prov in tqdm(provinces_to_lookup):
        found_region = find_region_for_province(prov)
        if found_region:
            province_region_map[prov] = found_region

    predicted_regions = df.loc[mask_missing_region, 'province'].map(province_region_map)

    print(f"mapping Dictionary created with {len(province_region_map)} entries.")
    if len(province_region_map) > 0:
        print(f"        Sample: {list(province_region_map.items())[:3]}")
    
    df.loc[mask_missing_region, 'region'] = predicted_regions

remaining_missing = (df['region'].isna() & df['province'].notna()).sum()
print(f"rows filled: {mask_missing_region.sum() - remaining_missing}")

rows missing Region but having province: 2


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.81it/s]

mapping Dictionary created with 1 entries.
        Sample: [('Ancona', 'Marche')]
rows filled: 2


### country

In [29]:
invalid_elems = df[df['country'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['country'].head(10))

nan_indexes = df.index[df['country'].isna()].tolist()
print(f"number of missing values in country: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: country, dtype: object
number of missing values in country: 34


In [30]:
unique_countries = sorted(df['country'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_countries)}")
print(unique_countries)

number of unique provinces: 1
['Italia']


as this column doesn't provide any useful information in this dataset as well, it can be eliminated.

### latitude

In [31]:
invalid_elems = df[df['latitude'].apply(check_type_validity, expected_type=float)]
print(invalid_elems['latitude'].head(10))

nan_indexes = df.index[df['latitude'].isna()].tolist()
print(f"number of missing values in latitude: {len(nan_indexes)}")

Series([], Name: latitude, dtype: float64)
number of missing values in latitude: 32


### longitude

In [32]:
invalid_elems = df[df['longitude'].apply(check_type_validity, expected_type=float)]
print(invalid_elems['longitude'].head(10))

nan_indexes = df.index[df['longitude'].isna()].tolist()
print(f"number of missing values in longitude: {len(nan_indexes)}")

Series([], Name: longitude, dtype: float64)
number of missing values in longitude: 32


let's check which rows have the combinations of region and province, and latitude and longitude. this because it should be possible to obtain region and province if we have latitude and longitude, and should be possible to optain the latitude and longitute if we have region and province (country is assumed to be Italy).

In [33]:
mask_has_address = (
    (df['province'].notna()) & 
    (df['region'].notna()) & 
    (df['latitude'].isna())
)

mask_has_coords = (
    (df['latitude'].notna()) & 
    (df['longitude'].notna()) & 
    (df['province'].isna())
)

print(f"rows to recover using address -> coordinates: {mask_has_address.sum()}")
print(f"rows to recover using coordinates -> address: {mask_has_coords.sum()}")

rows to recover using address -> coordinates: 0
rows to recover using coordinates -> address: 2


In [34]:
def get_osm_address(row):
    try:
        #pass coordinates as a string "lat, lon"
        query = f"{row['latitude']}, {row['longitude']}"
        location = reverse_osm(query, language='it')
        
        if location and location.raw.get('address'):
            addr = location.raw['address']
            #OSM mapping: 'state' -> Region, 'county' -> Province
            found_region = addr.get('state')
            found_province = addr.get('county', addr.get('city')) #fallback to city if county missing
            
            return pd.Series([found_province, found_region])
        return pd.Series([None, None])
    except:
        return pd.Series([None, None])

print("querying OpenStreetMap for missing address details...")
cols = ['province', 'region']
df.loc[mask_has_coords, cols] = df[mask_has_coords].progress_apply(
    get_osm_address, axis=1
).values

attempted_rows = df.loc[mask_has_coords]
success_rows = attempted_rows[attempted_rows['region'].notna()]
print(success_rows[['latitude', 'longitude', 'province', 'region']].head(10))

missing_province_count = df['province'].isna().sum()
missing_region_count = df['region'].isna().sum()

# 2. Print Singular Counts
print(f"rows missing province: {missing_province_count}")
print(f"rows missing region: {missing_region_count}")
print(f"rows missing both (region and province): {min(missing_province_count, missing_region_count)}")

querying OpenStreetMap for missing address details...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.07it/s]

     latitude  longitude province          region
6   45.080627   7.670717   Torino        Piemonte
89  44.803741  10.143004    Parma  Emilia-Romagna
rows missing province: 32
rows missing region: 32
rows missing both (region and province): 32


In [35]:
def get_wikidata_id(name): #function to find the ID
    search_url = "https://www.wikidata.org/w/api.php"
    params = {
        'action': 'wbsearchentities',
        'search': name,
        'language': 'it',
        'format': 'json',
        'limit': 1  #only top match
    }
    
    #User-Agent is mandatory
    headers = {'User-Agent': 'ArtistAnalysisBot/1.0 (student_project_test)'}

    try:
        response = requests.get(search_url, params=params, headers=headers, timeout=5)
        data = response.json()

        if data.get('success') and len(data.get('search', [])) > 0:
            return data['search'][0]['id']
    except Exception:
        return None
    return None

def get_location_details(qid): #function to find infos from the ID
    if not qid:
        return {'fetched_province': None, 'fetched_region': None}        

    sparql_url = "https://query.wikidata.org/sparql"   

    # Query: given this specific QID (wd:Q...), find the P131 (Admin Unit) chain
    query = f"""
    SELECT DISTINCT ?label ?typeLabel WHERE {{
      VALUES ?artist {{ wd:{qid} }}      

      #get Place: Birth (P19) OR Formation (P740)
      {{ ?artist wdt:P19 ?place. }} UNION {{ ?artist wdt:P740 ?place. }}      

      #walk up the hierarchy (P131) to find Italian Region (Q16110) or Province (Q15089)
      ?place wdt:P131* ?admin.
      ?admin wdt:P31 ?type.
      VALUES ?type {{ wd:Q16110 wd:Q15089 wd:Q15110 }}     

      #get the label
      ?admin rdfs:label ?label.
      FILTER(LANG(?label) = "it").

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "it". }}
    }}
    LIMIT 2
    """   

    headers = {'User-Agent': 'ArtistAnalysisBot/1.0 (student_project_test)'}

    try:
        response = requests.get(sparql_url, params={'format': 'json', 'query': query}, headers=headers, timeout=5)
        data = response.json()

        result = {'fetched_province': None, 'fetched_region': None}        

        #known regions for classification
        known_regions = [
            "Lombardia", "Lazio", "Campania", "Sicilia", "Veneto", "Piemonte", 
            "Emilia-Romagna", "Puglia", "Toscana", "Calabria", "Sardegna", "Liguria", 
            "Marche", "Abruzzo", "Friuli-Venezia Giulia", "Trentino-Alto Adige", 
            "Umbria", "Basilicata", "Molise", "Valle d'Aosta"
        ]

        for item in data['results']['bindings']:
            label = item['label']['value']
            #remove prefixes
            clean_label = (label.replace("città metropolitana di ", "")
                               .replace("Città metropolitana di ", "")
                               .replace("provincia di ", "")
                               .replace("Provincia di ", "").strip())

            if clean_label in known_regions:
                result['fetched_region'] = clean_label
            else:
                result['fetched_province'] = clean_label

        return result      

    except Exception:
        return {'fetched_province': None, 'fetched_region': None}


def fetch_artist_data_twostep(name):
    #get first ID
    qid = get_wikidata_id(name)
    #then get the location details
    if qid:
        return get_location_details(qid)
    return {'fetched_province': None, 'fetched_region': None}

validation_df = df[['id_author', 'name', 'province', 'region']].copy()

tqdm.pandas(desc="Fetching Data")
validation_results = validation_df['name'].progress_apply(fetch_artist_data_twostep)

validation_df = pd.concat([validation_df, pd.json_normalize(validation_results)], axis=1)

def check_match(row, col_db, col_fetched): #validation function
    val_db = row[col_db]
    val_fetch = row[col_fetched]   

    if pd.isna(val_db) or pd.isna(val_fetch):
        return "Inconclusive (Missing Data)"   

    # Normalize
    str_db = str(val_db).lower().strip()
    str_fetch = str(val_fetch).lower().strip()

    # Match logic
    if str_db == str_fetch:
        return "Match"

    if str_db in str_fetch or str_fetch in str_db:
        return "Match (Approx)"

    return "Mismatch"

validation_df['province_validation'] = validation_df.apply(check_match, args=('province', 'fetched_province'), axis=1)
validation_df['region_validation'] = validation_df.apply(check_match, args=('region', 'fetched_region'), axis=1)

#filter out inconclusive for clearer stats

valid_prov_checks = validation_df[validation_df['province_validation'] != "Inconclusive (Missing Data)"]
valid_reg_checks = validation_df[validation_df['region_validation'] != "Inconclusive (Missing Data)"]

print(f"\nProvince Validation (on {len(valid_prov_checks)} comparable rows):")
print(valid_prov_checks['province_validation'].value_counts())

print(f"\nRegion Validation (on {len(valid_reg_checks)} comparable rows):")
print(valid_reg_checks['region_validation'].value_counts())

matches = validation_df[validation_df['province_validation'].str.contains('Match', na=False)]
if not matches.empty:
    display(Markdown("Match Examples"))
    display(matches[['name', 'province', 'fetched_province']].head(5))

Fetching Data: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [01:47<00:00,  1.03s/it]


Province Validation (on 57 comparable rows):
province_validation
Match             47
Match (Approx)     9
Mismatch           1
Name: count, dtype: int64

Region Validation (on 57 comparable rows):
region_validation
Match    57
Name: count, dtype: int64


Match Examples

,name,province,fetched_province
1,achille lauro,Verona,Verona
5,babaman,Milano,Milano
7,bassi maestro,Milano,Milano
14,capo plaza,Salerno,Salerno
16,clementino,Avellino,Avellino


In [36]:
#using validation_df to fill df
rows_to_fill_prov = df['province'].isna() & validation_df['fetched_province'].notna()
rows_to_fill_reg = df['region'].isna() & validation_df['fetched_region'].notna()

count_prov = rows_to_fill_prov.sum()
count_reg = rows_to_fill_reg.sum()

#update df
df.loc[rows_to_fill_prov, 'province'] = validation_df.loc[rows_to_fill_prov, 'fetched_province'].values
df.loc[rows_to_fill_reg, 'region'] = validation_df.loc[rows_to_fill_reg, 'fetched_region'].values

print(f"successfully filled {count_prov} missing provinces.")
print(f"successfully filled {count_reg} missing regions.")

if count_prov > 0 or count_reg > 0:
    filled_indexes = rows_to_fill_prov | rows_to_fill_reg
    display(Markdown("Sample of enriched rows:"))
    
    cols_to_show = ['name', 'province', 'region']
    display(df.loc[filled_indexes, cols_to_show].head(10))

print(f"\nremaining missing provinces: {df['province'].isna().sum()}")
print(f"remaining missing regions: {df['region'].isna().sum()}")

successfully filled 7 missing provinces.
successfully filled 7 missing regions.


Sample of enriched rows:

,name,province,region
0,99 posse,Napoli,Campania
2,alfa,Milano,Lombardia
3,anna pepe,provincia della Spezia,Liguria
17,club dogo,Milano,Lombardia
22,dargen d_amico,Milano,Lombardia
42,guè pequeno,Milano,Lombardia
70,nerone,Roma Capitale,Lazio



remaining missing provinces: 25
remaining missing regions: 25


In [37]:
#get unique mapping of id_artist -> name_artist
track_names = df_tracks.drop_duplicates(subset=['id_artist'])

df = df.drop(columns=['name_artist', 'id_artist'], errors='ignore')
df = pd.merge(df, track_names, left_on='id_author', right_on='id_artist', how='left')

missing_loc_mask = df['province'].isna() | df['region'].isna()
has_track_name = df['name_artist'].notna()

#normalize names for valid comparison (lowercase, stripped)
name_main_norm = df['name'].fillna('').astype(str).str.lower().str.strip()
name_track_norm = df['name_artist'].fillna('').astype(str).str.lower().str.strip()
is_different_name = name_main_norm != name_track_norm

candidates_mask = missing_loc_mask & has_track_name & is_different_name
candidates = df[candidates_mask].copy()

print(f"found {len(candidates)} artists with missing location and a DIFFERENT name in tracks dataset.")

def fetch_missing_data(name):
    """Fetch location data for an artist name"""
    result = fetch_artist_data_twostep(name)
    return {
        'new_province': result['fetched_province'],
        'new_region': result['fetched_region']
    }


if not candidates.empty:
    print("candidates for alternative search:", candidates['name_artist'].unique())

    tqdm.pandas(desc="Retry with different versions of name")
    
    retry_results = candidates['name_artist'].progress_apply(fetch_missing_data) #name_artist as the argument
    
    retry_df = pd.json_normalize(retry_results)
    retry_df.index = candidates.index
    
    fill_prov = df['province'].isna() & retry_df['new_province'].notna()
    fill_reg = df['region'].isna() & retry_df['new_region'].notna()
    
    if fill_prov.sum() > 0:
        df.loc[fill_prov, 'province'] = retry_df.loc[fill_prov, 'new_province'].values
    if fill_reg.sum() > 0:
        df.loc[fill_reg, 'region'] = retry_df.loc[fill_reg, 'new_region'].values
    
    print(f"recovered {fill_prov.sum()} provinces.")
    print(f"recovered {fill_reg.sum()} regions.")
    
    # Show samples if any found
    if fill_prov.sum() > 0 or fill_reg.sum() > 0:
        recovered_mask = fill_prov | fill_reg
        cols_to_show = ['name', 'name_artist', 'province', 'region']
        display(df.loc[recovered_mask, cols_to_show].head())
else:
    print("No useful alternative names found for the missing rows.")

cols_to_drop = ['id_artist', 'name_artist']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

missing_prov_count = df['province'].isna().sum()
missing_reg_count = df['region'].isna().sum()
print(f"\nmissing provinces: {missing_prov_count}")
print(f"missing regions: {missing_reg_count}")

found 5 artists with missing location and a DIFFERENT name in tracks dataset.
candidates for alternative search: ['Joey (ITA)' 'Highsnob' 'M¥SS KETA' '’O Zulù' 'Samuel Costa']


Retry with different versions of name: 100%|█████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.15it/s]

recovered 3 provinces.
recovered 3 regions.


,name,name_artist,province,region
61,mike24,Highsnob,Avellino,Campania
62,miss keta,M¥SS KETA,Milano,Lombardia
75,o zulù,’O Zulù,Napoli,Campania



missing provinces: 22
missing regions: 22


In [38]:
def fix_artist_geography(df):
    """
    Applies manual geographic research to the artist dataset.
    Operates directly on the provided dataframe 'df'.
    """
    
    #knowledge derived from manual research
    verified_geo_data = {
        'beba': {'province': 'Torino', 'region': 'Piemonte'},
        'bigmama': {'province': 'Avellino', 'region': 'Campania'},
        'brusco': {'province': 'Roma', 'region': 'Lazio'},
        'bushwaka': {'province': 'La Spezia', 'region': 'Liguria'},
        'caneda': {'province': 'Milano', 'region': 'Lombardia'},
        'colle der fomento': {'province': 'Roma', 'region': 'Lazio'},
        'cor veleno': {'province': 'Roma', 'region': 'Lazio'},
        'dark polo gang': {'province': 'Roma', 'region': 'Lazio'},
        'doll kill': {'province': 'Sassari', 'region': 'Sardegna'},
        'eva rea': {'province': 'Catania', 'region': 'Sicilia'},
        'hindaco': {'province': 'Milano', 'region': 'Lombardia'},
        'joey funboy': {'province': 'Bolzano', 'region': 'Trentino-Alto Adige'},
        'johnny marsiglia': {'province': 'Palermo', 'region': 'Sicilia'},
        'miss simpatia': {'province': 'Ancona', 'region': 'Marche'}, # Falconara M. is in Ancona
        'mistico': {'province': 'Milano', 'region': 'Lombardia'},
        'priestess': {'province': 'Bari', 'region': 'Puglia'}, # Locorotondo is in Bari
        'samuel heron': {'province': 'La Spezia', 'region': 'Liguria'},
        'shiva': {'province': 'Milano', 'region': 'Lombardia'}, # Legnano is in Milano
        'skioffi': {'province': 'Frosinone', 'region': 'Lazio'},
        'sottotono': {'province': 'Milano', 'region': 'Lombardia'},
        'yendry': {'province': 'Torino', 'region': 'Piemonte'}
    }
    
    print(f"manual update for {len(verified_geo_data)} verified entities")
    
    updates_count = 0
    
    #iterate through the main dataframe
    for index, row in df.iterrows():
        #normalize artist name for lookup (using the 'name' column, not 'artist')
        artist_key = str(row['name']).lower().strip()
        
        #handle 'yendry' with potential cyrillic confusion if present
        if 'ye' in artist_key and 'dry' in artist_key: 
             artist_key = 'yendry'

        if artist_key in verified_geo_data:
            data = verified_geo_data[artist_key]
            
            #update columns with full names in the main dataframe
            df.at[index, 'province'] = data['province']
            df.at[index, 'region'] = data['region']
            updates_count += 1
            
    return df

#execute the fix on the global 'df'
df = fix_artist_geography(df)

manual update for 21 verified entities
